# 🛍️ Sistema de Recomendación de Productos (Clustering & K-Means)

> Sistema de recomendación híbrido no supervisado para e-commerce basado en **segmentación de catálogo mediante K-Means**, **reducción dimensional con PCA** y **búsqueda por similitud euclidiana intra-cluster**.

---

## 🎯 Objetivo de Negocio

Incrementar el *cross-selling* y la tasa de conversión recomendando a los compradores productos de la **misma categoría y con atributos comerciales similares** (precio, flete, calificación y nivel de ventas), evitando mezclar gamas dispares (ej. productos ultra-económicos con productos premium).

```mermaid
flowchart LR
    A[Datasets Relacionales Olist] --> B[src.common.data]
    B --> C[src.recommendations.features]
    C --> D[K-Means Clustering + PCA]
    D --> E[Similitud Euclidiana Intra-Cluster]
    E --> F[src.recommendations.viz]
```


In [ ]:
# Librerías estándar y de terceros
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Módulos del proyecto (instalados en modo editable via pyproject.toml)
from src.common.data import load_recommendations_raw_data
from src.recommendations.features import (
    FEATURE_COLS,
    build_product_features,
    compute_pca,
    evaluate_clustering_metrics,
    find_optimal_clusters,
    recomendar_productos,
    scale_features,
    train_kmeans_clusters,
)
from src.recommendations.viz import (
    plot_clusters_pca_2d,
    plot_elbow_and_silhouette,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 1. Carga y Unión de Datasets

Ingesta de las 3 tablas base (productos, items de órdenes y reseñas) mediante el cargador centralizado de `src.common.data`.

In [ ]:
raw_data = load_recommendations_raw_data()
products_df = raw_data["products"]
order_items_df = raw_data["items"]
reviews_df = raw_data["reviews"]

print(f"Productos cargados: {len(products_df):,}")
print(f"Items de órdenes cargados: {len(order_items_df):,}")
print(f"Reseñas cargadas: {len(reviews_df):,}")


## 2. Ingeniería de Características (Feature Engineering)

Construcción del dataset a nivel de producto mediante `build_product_features`:
- Agrega precio promedio, flete promedio, total de ventas y calificación de reseñas por producto.
- Imputa reseñas faltantes con la media global.
- Calcula el ratio de flete sobre precio con recorte al percentil 99.9.
- Aplica transformaciones $\log(1 + x)$ para estabilizar variables asimétricas.
- Asocia la categoría de producto.

In [ ]:
df_model = build_product_features(products_df, order_items_df, reviews_df)
print(f"Productos consolidados para modelado: {len(df_model):,}")
df_model.head()


## 3. Escalado y Proyección Dimensional (PCA 2D)

Estandarización de las 5 variables numéricas y proyección en dos componentes principales para diagnóstico visual.

In [ ]:
scaled_features, scaler = scale_features(df_model, FEATURE_COLS)
pca_data, pca = compute_pca(scaled_features, n_components=2)

print(f"Varianza explicada por componente PCA: {pca.explained_variance_ratio_}")
print(f"Varianza total explicada en 2D: {pca.explained_variance_ratio_.sum():.2%}")


## 4. Selección del Número Óptimo de Clusters (K)

Evaluación del rango $K \in [2, 7]$ mediante el Método del Codo (Inertia) y el Silhouette Score sobre una muestra reproducible.

In [ ]:
inertias, silhouettes, inertia_vals, sil_vals, best_k = find_optimal_clusters(
    scaled_features, k_range=range(2, 8), sample_size=10000, random_state=42
)

plot_elbow_and_silhouette(range(2, 8), inertias, silhouettes)
print(f"K óptimo seleccionado por métricas: K={best_k} (Silhouette = {sil_vals[best_k]:.4f})")


## 5. Entrenamiento Final y Perfilado de Clusters (K=5)

Ajuste del modelo K-Means con $K=5$, generación de perfiles estadísticos promedio y asignación de etiquetas semánticas de negocio.

In [ ]:
df_model, cluster_profile, kmeans = train_kmeans_clusters(
    df_model, scaled_features, n_clusters=best_k, random_state=42
)

print("=== Perfil Promedio por Cluster (8 Columnas) ===")
ordered_cols = [
    "cluster_name",
    "price",
    "freight_value",
    "freight_ratio",
    "review_score",
    "total_sales",
    "count",
    "pct",
]
display(cluster_profile[ordered_cols].round(2))

plot_clusters_pca_2d(pca_data, df_model["cluster_name"])


## 6. Función del Sistema de Recomendación Híbrido

Búsqueda de los $N$ productos más similares dentro de la misma categoría y gama mediante distancia euclidiana en el espacio vectorial escalado.

In [ ]:
sample_product_id = df_model[df_model["product_category_name"] == "automotivo"].iloc[0]["product_id"]
recs_result = recomendar_productos(sample_product_id, df_model, scaled_features, n_recs=5)
display(recs_result)


## 7. Evaluación Formal del Modelo de Clustering

Cálculo de métricas de calidad de agrupamiento intrínsecas: índice Calinski-Harabasz (dispersión entre clusters) e índice Davies-Bouldin (separabilidad y compacidad).

In [ ]:
metrics = evaluate_clustering_metrics(scaled_features, df_model["cluster"])

print(f"Métricas de Evaluación Global (K={best_k}):")
print(f"- Calinski-Harabasz Index: {metrics['calinski_harabasz_score']:.2f} (Mayor es mejor: alta dispersión entre clusters)")
print(f"- Davies-Bouldin Index:    {metrics['davies_bouldin_score']:.4f} (Menor es mejor: clusters compactos y separados)")
